In [1]:
import pyscrew
import matplotlib.pyplot as plt
import pandas as pd
from neuralop.models import FNO
from sklearn.preprocessing import OneHotEncoder

In [2]:
# Load data from the thread degradation scenario (s02)
data_v2 = pyscrew.get_data(scenario="s02", handle_duplicates="first", handle_missings="mean",cache_dir="~/.cache/pyscrew", force_download=False, target_length=800)

# Access the measurements and labels
x_values_v2 = data_v2["torque_values"]
y_values_v2 = data_v2["class_values"]
usage_values_v2 = data_v2["workpiece_usage"]

2026-05-20 15:57:01 - INFO - pyscrew.main - Starting data retrieval for scenario: s02 (surface-friction)
2026-05-20 15:57:01 - INFO - pyscrew.pipeline.loading - Using cache directory (absolute): C:\Users\Patrick\.cache\pyscrew
2026-05-20 15:57:01 - INFO - pyscrew.pipeline.loading - Beginning data extraction for scenario 's02_variations-in-surface-friction.zip' (force=False)
2026-05-20 15:57:01 - INFO - pyscrew.pipeline.loading - Verifying MD5 checksum for 's02_variations-in-surface-friction.zip'...
2026-05-20 15:57:01 - INFO - pyscrew.pipeline.loading - Checksum verification successful for 's02_variations-in-surface-friction.zip' (MD5: 0bc948a6e8c6e83f72dbe36973131558)
2026-05-20 15:57:01 - INFO - pyscrew.pipeline.loading - Using existing verified file 's02_variations-in-surface-friction.zip' at: C:\Users\Patrick\.cache\pyscrew\archives\s02_variations-in-surface-friction.zip
2026-05-20 15:57:02 - INFO - pyscrew.pipeline.loading - Using existing extracted data for scenario 's02_variatio

2026-05-20 15:59:09 - INFO - pyscrew.pipeline.transformers.handle_missings - Completed missing interpolation using 'mean' method (interval=0.0012)
2026-05-20 15:59:09 - INFO - pyscrew.pipeline.transformers.handle_missings - Processed 12,500 series with 8,619,982 total points
2026-05-20 15:59:09 - INFO - pyscrew.pipeline.transformers.handle_missings - Found gaps - min: 0.0012s, max: 0.1128s, avg: 0.0013s
2026-05-20 15:59:09 - INFO - pyscrew.pipeline.transformers.handle_missings - Added 484,205 points (+5.62% of total)
2026-05-20 15:59:09 - INFO - pyscrew.pipeline.transformers.handle_missings - Average 38.7 points added per series
2026-05-20 15:59:09 - INFO - pyscrew.pipeline.transformers.handle_lengths - Starting to apply equal lengths.


2026-05-20 15:59:09 - INFO - pyscrew.pipeline.transformers.handle_lengths - - 'target_length' : 800
2026-05-20 15:59:09 - INFO - pyscrew.pipeline.transformers.handle_lengths - - 'padding_value' : 0.0
2026-05-20 15:59:09 - INFO - pyscrew.pipeline.transformers.handle_lengths - - 'padding_position' : post
2026-05-20 15:59:09 - INFO - pyscrew.pipeline.transformers.handle_lengths - - 'cutoff_position' : post
2026-05-20 15:59:11 - INFO - pyscrew.pipeline.transformers.handle_lengths - Finished applying equal lengths to the screw driving data.
2026-05-20 15:59:11 - INFO - pyscrew.pipeline.transformers.handle_lengths - - Total screw runs loaded:	12500
2026-05-20 15:59:11 - INFO - pyscrew.pipeline.transformers.handle_lengths - - Average change of length:	728.33 -> 800.00
2026-05-20 15:59:11 - INFO - pyscrew.pipeline.transformers.handle_lengths - - Total points before normalization:	9,104,187
2026-05-20 15:59:11 - INFO - pyscrew.pipeline.transformers.handle_lengths - - Total points after normaliz

In [3]:
data_v2_df = pd.DataFrame(data_v2)

In [4]:
data_v2_df.head()

,time_values,torque_values,angle_values,gradient_values,step_values,class_values,workpiece_location,workpiece_usage,workpiece_result,scenario_condition,scenario_exception
0,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.067, 0.077, 0.126, 0.087, 0.089, 0.097, 0.0...","[0.5, 1.25, 2.25, 3.75, 5.0, 6.25, 7.5, 8.75, ...","[0.0, 0.0, 0.0298, 0.0214, 0.0064, 0.0023, -0....","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,0,OK,normal,0
1,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.005, 0.008, 0.061, 0.069, 0.104, 0.124, 0....","[0.0, 0.25, 0.75, 1.5, 2.5, 4.0, 5.25, 6.5, 7....","[0.0, 0.0, 0.0, 0.0, 0.0287, 0.0282, 0.0091, 0...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,0,OK,normal,0
2,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, 0.01, 0.039, 0.099, 0.102, 0.087, 0.0...","[0.0, 0.5, 1.25, 2.25, 3.5, 4.75, 6.0, 7.5, 8....","[0.0, 0.0, 0.0, 0.0196, 0.0223, 0.0211, 0.0096...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,1,OK,normal,0
3,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.081, 0.059, 0.12, 0.077, 0.043, 0.059, 0.06...","[0.75, 1.75, 2.75, 4.0, 5.25, 6.5, 7.75, 9.25,...","[0.0, 0.0, 0.0261, 0.0207, 0.0, -0.0036, -0.00...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,1,OK,normal,0
4,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, -0.0012, 0.0006, 0.0024, 0.0042, 0.00...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 1.5, 2.5, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.025...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,2,OK,normal,0


In [5]:
"""
@author: Modified code from Zongyi Li, modified by QueensGambit
This file is the Fourier Neural Operator for 1D problem such as the (time-independent) Burgers equation discussed in Section 5.1 in the [paper](https://arxiv.org/pdf/2010.08895.pdf).
"""

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parameter import Parameter
import matplotlib.pyplot as plt

import operator
from functools import reduce
from functools import partial
from timeit import default_timer
#from fourier-neural-operator.utilities3 import *

#from Adam import Adam

torch.manual_seed(0)
np.random.seed(0)

In [6]:
################################################################
#  1d fourier layer
################################################################
class SpectralConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1):
        super(SpectralConv1d, self).__init__()

        """
        1D Fourier layer. It does FFT, linear transform, and Inverse FFT.    
        """

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1  #Number of Fourier modes to multiply, at most floor(N/2) + 1

        self.scale = (1 / (in_channels*out_channels))
        self.weights1 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, dtype=torch.cfloat))

    # Complex multiplication
    def compl_mul1d(self, input, weights):
        # (batch, in_channel, x ), (in_channel, out_channel, x) -> (batch, out_channel, x)
        return torch.einsum("bix,iox->box", input, weights)

    def forward(self, x):
        batchsize = x.shape[0]
        #Compute Fourier coeffcients up to factor of e^(- something constant)
        x_ft = torch.fft.rfft(x)

        # Multiply relevant Fourier modes
        out_ft = torch.zeros(batchsize, self.out_channels, x.size(-1)//2 + 1,  device=x.device, dtype=torch.cfloat)
        out_ft[:, :, :self.modes1] = self.compl_mul1d(x_ft[:, :, :self.modes1], self.weights1)

        #Return to physical space
        x = torch.fft.irfft(out_ft, n=x.size(-1))
        return x

class FNO1d(nn.Module):
    def __init__(self, modes, width, n_classes):
        super(FNO1d, self).__init__()

        """
        The overall network. It contains 4 layers of the Fourier layer.
        1. Lift the input to the desire channel dimension by self.fc0 .
        2. 4 layers of the integral operators u' = (W + K)(u).
            W defined by self.w; K defined by self.conv .
        3. Project from the channel space to the output space by self.fc1 and self.fc2 .
        
        input: the solution of the initial condition and location (a(x), x)
        input shape: (batchsize, x=s, c=2)
        output: the solution of a later timestep
        output shape: (batchsize, x=s, c=1)
        """

        self.modes1 = modes
        self.width = width
        self.padding = 2 # pad the domain if input is non-periodic
        self.fc0 = nn.Linear(2, self.width) # input channel is 2: (a(x), x)

        self.conv0 = SpectralConv1d(self.width, self.width, self.modes1)
        self.conv1 = SpectralConv1d(self.width, self.width, self.modes1)
        self.conv2 = SpectralConv1d(self.width, self.width, self.modes1)
        self.conv3 = SpectralConv1d(self.width, self.width, self.modes1)
        self.w0 = nn.Conv1d(self.width, self.width, 1)
        self.w1 = nn.Conv1d(self.width, self.width, 1)
        self.w2 = nn.Conv1d(self.width, self.width, 1)
        self.w3 = nn.Conv1d(self.width, self.width, 1)

        self.fc1 = nn.Linear(self.width, 128)
        self.fc2 = nn.Linear(128, n_classes)

    def forward(self, x):
        grid = self.get_grid(x.shape, x.device)
        x = torch.cat((x, grid), dim=-1)
        x = self.fc0(x)
        x = x.permute(0, 2, 1)
        # x = F.pad(x, [0,self.padding]) # pad the domain if input is non-periodic

        x1 = self.conv0(x)
        x2 = self.w0(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv1(x)
        x2 = self.w1(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv2(x)
        x2 = self.w2(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv3(x)
        x2 = self.w3(x)
        x = x1 + x2

        # 1. Global Pooling: Average across the 1000 time steps
        # This collapses the sequence dimension
        x = torch.mean(x, dim=-1) # New shape: (batch, width)

        # x = x[..., :-self.padding] # pad the domain if input is non-periodic
        #x = x.permute(0, 2, 1)
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        return x

    def get_grid(self, shape, device):
        batchsize, size_x = shape[0], shape[1]
        gridx = torch.tensor(np.linspace(0, 1, size_x), dtype=torch.float)
        gridx = gridx.reshape(1, size_x, 1).repeat([batchsize, 1, 1])
        return gridx.to(device)
    
class FNO2d(nn.Module):
    def __init__(self, modes, width, n_classes, input_channels = 1):
        super(FNO2d, self).__init__()

        """
        The overall network. It contains 4 layers of the Fourier layer.
        1. Lift the input to the desire channel dimension by self.fc0 .
        2. 4 layers of the integral operators u' = (W + K)(u).
            W defined by self.w; K defined by self.conv .
        3. Project from the channel space to the output space by self.fc1 and self.fc2 .
        
        input: the solution of the initial condition and location (a(x), x)
        input shape: (batchsize, x=s, c=2)
        output: the solution of a later timestep
        output shape: (batchsize, x=s, c=1)
        """

        self.modes1 = modes
        self.width = width
        self.padding = 2 # pad the domain if input is non-periodic
        self.fc0 = nn.Linear(input_channels +1, self.width) # input channel is flexibel for more features e.g. torque + angle

        self.conv0 = SpectralConv1d(self.width, self.width, self.modes1)
        self.conv1 = SpectralConv1d(self.width, self.width, self.modes1)
        self.conv2 = SpectralConv1d(self.width, self.width, self.modes1)
        self.conv3 = SpectralConv1d(self.width, self.width, self.modes1)
        self.w0 = nn.Conv1d(self.width, self.width, 1)
        self.w1 = nn.Conv1d(self.width, self.width, 1)
        self.w2 = nn.Conv1d(self.width, self.width, 1)
        self.w3 = nn.Conv1d(self.width, self.width, 1)

        self.fc1 = nn.Linear(self.width, 128)
        self.fc2 = nn.Linear(128, n_classes)

    def forward(self, x):
        grid = self.get_grid(x.shape, x.device)
        x = torch.cat((x, grid), dim=-1)
        x = self.fc0(x)
        x = x.permute(0, 2, 1)
        # x = F.pad(x, [0,self.padding]) # pad the domain if input is non-periodic

        x1 = self.conv0(x)
        x2 = self.w0(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv1(x)
        x2 = self.w1(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv2(x)
        x2 = self.w2(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv3(x)
        x2 = self.w3(x)
        x = x1 + x2

        # 1. Global Pooling: Average across the 1000 time steps
        # This collapses the sequence dimension
        x = torch.mean(x, dim=-1) # New shape: (batch, width)

        # x = x[..., :-self.padding] # pad the domain if input is non-periodic
        #x = x.permute(0, 2, 1)
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        return x

    def get_grid(self, shape, device):
        batchsize, size_x = shape[0], shape[1]
        gridx = torch.tensor(np.linspace(0, 1, size_x), dtype=torch.float)
        gridx = gridx.reshape(1, size_x, 1).repeat([batchsize, 1, 1])
        return gridx.to(device)

In [9]:
################################################################
#  configurations
################################################################
ntrain = 1000
ntest = 100

sub = 2**3 #subsampling rate
h = 2**13 // sub #total grid size divided by the subsampling rate
s = h

batch_size = 20
learning_rate = 0.001

epochs = 500
step_size = 50
gamma = 0.5

modes = 16
width = 64

In [10]:
len(data_v2_df["torque_values"])

12500

In [11]:
x_data = np.array(data_v2_df["torque_values"].tolist())
y_data = np.array(data_v2_df["class_values"].tolist())

In [12]:
y_data.shape

(12500,)

In [13]:
x_data.shape

(12500, 800)

In [14]:
from sklearn.utils import shuffle 

In [15]:
################################################################
# read data
################################################################

# Data is of the shape (number of samples, grid size)
#dataloader = MatReader('data/burgers_data_R10.mat')
#x_data = dataloader.read_field('a')[:,::sub]
#y_data = dataloader.read_field('u')[:,::sub]
x_data = np.array(data_v2_df['torque_values'].tolist())
y_data = np.array(data_v2_df['class_values'].tolist())
usage = np.array(data_v2_df['workpiece_usage'].tolist())

# Reshape is required because OneHotEncoder expects a 2D array (samples, features)
data_reshaped = y_data.reshape(-1, 1)

# Initialize and transform
encoder = OneHotEncoder(sparse_output=False) # Use sparse=False to get a dense numpy array
one_hot = encoder.fit_transform(data_reshaped)

print(one_hot)
# To see the labels corresponding to the columns:
print(encoder.get_feature_names_out())

split_ratio = 0.65
n_samples = len(x_data)
split_idx = int(n_samples * split_ratio)

X_s, y_s = shuffle(x_data, one_hot)

x_train = X_s[:split_idx,:]
y_train = y_s[:split_idx,:]
x_test = X_s[split_idx:,:]
y_test = y_s[split_idx:,:]

x_train_tensor = torch.from_numpy(x_train).float().unsqueeze(-1)
y_train_tensor = torch.from_numpy(y_train).float().argmax(axis=1)

x_test_tensor = torch.from_numpy(x_test).float().unsqueeze(-1)
y_test_tensor = torch.from_numpy(y_test).float().argmax(axis=1)

#x_train = x_train.reshape(ntrain,s,1)
#x_test = x_test.reshape(ntest,s,1)

#train_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)
#test_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_test_tensor, y_test_tensor), batch_size=batch_size, shuffle=False)

# model
model = FNO1d(modes, width, n_classes=8).cuda()
#model = FNO(n_modes=(64, 64),
#               hidden_channels=64,
#               in_channels=1,
#               out_channels=1)

#print(count_params(model))
model_2d = FNO2d(modes, width, n_classes=8, input_channels=2).cuda()

[[1. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 1.]]
['x0_001_control-group' 'x0_101_used-upper-workpiece'
 'x0_201_lubricant-water' 'x0_202_lubricant-oil-based'
 'x0_301_sanding-coarse' 'x0_302_sanding-fine' 'x0_401_plastic-adhesive'
 'x0_402_surface-chipped']


In [16]:
x_train.shape

(8125, 800)

In [17]:
y_train.shape

(8125, 8)

In [18]:
y_test_tensor.shape

torch.Size([4375])

In [19]:
n_classes = y_train.shape[1]
n_classes

8

In [20]:
#loss function with rel/abs Lp loss
class LpLoss(object):
    def __init__(self, d=2, p=2, size_average=True, reduction=True):
        super(LpLoss, self).__init__()

        #Dimension and Lp-norm type are postive
        assert d > 0 and p > 0

        self.d = d
        self.p = p
        self.reduction = reduction
        self.size_average = size_average

    def abs(self, x, y):
        num_examples = x.size()[0]

        #Assume uniform mesh
        h = 1.0 / (x.size()[1] - 1.0)

        all_norms = (h**(self.d/self.p))*torch.norm(x.view(num_examples,-1) - y.view(num_examples,-1), self.p, 1)

        if self.reduction:
            if self.size_average:
                return torch.mean(all_norms)
            else:
                return torch.sum(all_norms)

        return all_norms

    def rel(self, x, y):
        num_examples = x.size()[0]

        diff_norms = torch.norm(x.reshape(num_examples,-1) - y.reshape(num_examples,-1), self.p, 1)
        y_norms = torch.norm(y.reshape(num_examples,-1), self.p, 1)

        if self.reduction:
            if self.size_average:
                return torch.mean(diff_norms/y_norms)
            else:
                return torch.sum(diff_norms/y_norms)

        return diff_norms/y_norms

    def __call__(self, x, y):
        return self.rel(x, y)

In [21]:
import torch
from torchmetrics.classification import MulticlassAccuracy, MulticlassF1Score

# Assuming n_classes = 8 based on your y_train.shape
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

acc_metric = MulticlassAccuracy(num_classes=n_classes).to(device)
f1_metric = MulticlassF1Score(num_classes=n_classes).to(device)

In [ ]:

################################################################
# training and evaluation
################################################################
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

myloss = LpLoss(size_average=False)

#early stopping parameter
patience = 30
best_test_f1 = -np.inf
patience_counter=0
best_model_state=None
for ep in range(epochs):
    model.train()
    t1 = default_timer()
    train_cross_entropy = 0
    #train_l2 = 0
    for x, y in train_loader:
        x, y = x.cuda(), y.cuda()
        optimizer.zero_grad()
        out = model(x)
        #mse = F.mse_loss(out.view(batch_size, -1), y.view(batch_size, -1), reduction='mean')
        cross_entropy = F.cross_entropy(out, y)
        #cross_entropy = F.cross_entropy(out.view(batch_size, -1), y.view(batch_size, -1))
        #l2 = myloss(out.view(batch_size, -1), y.view(batch_size, -1))
        #l2.backward() # use the l2 relative loss
        cross_entropy.backward()
        optimizer.step()
        train_cross_entropy += cross_entropy.item()
        #train_l2 += l2.item()
        # 3. Update Metrics
        acc_metric.update(out, y)
        f1_metric.update(out, y)
    
    # 4. Compute final values for the epoch
    epoch_acc = acc_metric.compute()
    epoch_f1 = f1_metric.compute()
    print(f"Train Accuracy: {epoch_acc:.4f}, F1 Score: {epoch_f1:.4f}")

    acc_metric.reset()  # <--- RESET FOR TEST PHASE
    f1_metric.reset()

    scheduler.step()
    model.eval()
    test_cross_entropy = 0.0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.cuda(), y.cuda()

            out = model(x)
            test_cross_entropy = F.cross_entropy(out, y)  # ruft nur den letzten batch auf und test_cross_entropy wird üebrschrieben udn nciht aufsummiert
            #test_l2 += myloss(out.view(batch_size, -1), y.view(batch_size, -1)).item()

            # 3. Update Metrics
            acc_metric.update(out, y)
            f1_metric.update(out, y)
    
    # 4. Compute final values for the epoch
    epoch_acc = acc_metric.compute()
    epoch_f1 = f1_metric.compute()
    
    print(f"Test Accuracy: {epoch_acc:.4f}, F1 Score: {epoch_f1:.4f}")
    
    train_cross_entropy /= len(train_loader) # hier wird mit test_cross_entropy = F.cross_entropy(out, y) lediglich der letzte batchaufgerufen
    #train_l2 /= ntrain
    #test_l2 /= ntest
    t2 = default_timer()
    print(ep, t2-t1, train_cross_entropy, test_cross_entropy)
    if epoch_f1 > best_test_f1:
        best_test_f1 = epoch_f1
        best_model_state = model.state_dict().copy() 
        patience_counter = 0
        print(f"Neue bestes f1: {best_test_f1:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopped da keine Verbesserung nach {ep+1} epochen")
            if best_model_state is not None: 
                model.load_state_dict(best_model_state)
            break

# torch.save(model, 'model/ns_fourier_burgers')
pred = torch.zeros(y_test.shape)
index = 0
test_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_test_tensor, y_test_tensor), batch_size=1, shuffle=False)
with torch.no_grad():
    for x, y in test_loader:
        test_l2 = 0
        x, y = x.cuda(), y.cuda()

        out = model(x) #.view(-1)
        pred[index] = out

        test_cross_entropy = F.cross_entropy(out, y)
        #test_l2 += myloss(out.view(1, -1), y.view(1, -1)).item()
        print(index, test_cross_entropy)
        index = index + 1

# scipy.io.savemat('pred/burger_test.mat', mdict={'pred': pred.cpu().numpy()})

Train Accuracy: 0.5746, F1 Score: 0.5726
Test Accuracy: 0.4116, F1 Score: 0.4029
0 12.418911300002947 0.7719479263268173 tensor(0.1891, device='cuda:0')
Neue bestes f1: 0.4029
Train Accuracy: 0.5918, F1 Score: 0.5884
Test Accuracy: 0.3927, F1 Score: 0.3860
1 12.859266999999818 0.719230580864433 tensor(0.5658, device='cuda:0')
Train Accuracy: 0.5907, F1 Score: 0.5912
Test Accuracy: 0.4131, F1 Score: 0.4088
2 12.572723199999018 0.6802452021353954 tensor(0.2162, device='cuda:0')
Neue bestes f1: 0.4088
Train Accuracy: 0.6162, F1 Score: 0.6152
Test Accuracy: 0.4034, F1 Score: 0.4039
3 14.016685200003849 0.6260219846076402 tensor(0.5317, device='cuda:0')
Train Accuracy: 0.6219, F1 Score: 0.6240
Test Accuracy: 0.4095, F1 Score: 0.4032
4 13.682897700004105 0.6012273954744713 tensor(0.2195, device='cuda:0')
Train Accuracy: 0.6365, F1 Score: 0.6349
Test Accuracy: 0.4113, F1 Score: 0.4083
5 13.578654799996002 0.5755950745918241 tensor(0.3089, device='cuda:0')
Train Accuracy: 0.6463, F1 Score: 0.6

# HIER einmal testen - predfilter einfügen zur auswertung am ende und earlystopping einbinden. Ach ebenso die folds auszulesen.

In [22]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
#pme hot encodings labels für skfold
y_labels = one_hot.argmax(axis=1)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

for fold_num, (train_idx, test_idx) in enumerate(skf.split(x_data, y_labels),start=1):
    print(f"Fold {fold_num}von 5")
    
    #split basierend auf cv skfold
    X_train_fold = x_data[train_idx]
    X_test_fold = x_data[test_idx]
    y_train_fold = one_hot[train_idx]
    y_test_fold = one_hot[test_idx]
    usage_test_fold = usage[test_idx]
    
    #standardisieren
    scaler = StandardScaler()
    X_train_fold = scaler.fit_transform(X_train_fold.reshape(X_train_fold.shape[0], -1)).reshape(X_train_fold.shape)
    X_test_fold = scaler.transform(X_test_fold.reshape(X_test_fold.shape[0], -1)).reshape(X_test_fold.shape) 
    
    #konvertieren zu tensoren
    x_train_tensor = torch.from_numpy(X_train_fold).float().unsqueeze(-1)
    y_train_tensor = torch.from_numpy(y_train_fold).float().argmax(axis=1)
    x_test_tensor = torch.from_numpy(X_test_fold).float().unsqueeze(-1)
    y_test_tensor = torch.from_numpy(y_test_fold).float().argmax(axis=1)
    
    #train und testset als datasets
    train_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)
    test_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_test_tensor, y_test_tensor), batch_size=batch_size, shuffle=False)
    #model initialisiert, weights und lrscheduler
    model = FNO1d(modes, width, n_classes=n_classes).cuda()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)
    
    #early stopping
    patience = 30
    best_test_f1 = -np.inf
    patience_counter = 0
    best_model_state = None
    
    #Trainingsloop über trainingsdaten
    for ep in range(epochs):
        model.train()
        t1 = default_timer()
        train_cross_entropy = 0
        
        for x, y in train_loader:
            x, y = x.cuda(), y.cuda()
            optimizer.zero_grad()
            out = model(x)
            cross_entropy = F.cross_entropy(out, y)
            cross_entropy.backward()
            optimizer.step()
            train_cross_entropy += cross_entropy.item()
            acc_metric.update(out, y)
            f1_metric.update(out, y)
        
        epoch_acc = acc_metric.compute()
        epoch_f1 = f1_metric.compute()
        print(f"Epoch {ep+1}: Train Acc={epoch_acc:.4f}, F1={epoch_f1:.4f}")
        
        acc_metric.reset()
        f1_metric.reset()
        
        #testing loop
        scheduler.step()
        model.eval()
        test_cross_entropy = 0.0
        
        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.cuda(), y.cuda()
                out = model(x)
                test_cross_entropy += F.cross_entropy(out, y).item()
                acc_metric.update(out, y)
                f1_metric.update(out, y)
        
        test_cross_entropy /= len(test_loader)
        epoch_acc = acc_metric.compute()
        epoch_f1 = f1_metric.compute()
        
        print(f"Test  Acc={epoch_acc:.4f}, F1={epoch_f1:.4f}")
        
        train_cross_entropy /= len(train_loader)
        t2 = default_timer()
        
        # Early Stopping
        if float(epoch_f1) > best_test_f1:
            best_test_f1 = float(epoch_f1)
            best_model_state = model.state_dict().copy()
            patience_counter = 0
            print(f"Neue beste F1: {best_test_f1:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"____Early Stop nach Epoch {ep+1}")
                if best_model_state is not None:
                    model.load_state_dict(best_model_state)
                break
        
        acc_metric.reset()
        f1_metric.reset()
    
    #Preidiction auf testset
    y_pred_fold = []
    model.eval()
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.cuda(), y.cuda()
            out = model(x)
            preds = out.argmax(dim=1).cpu().numpy()
            y_pred_fold.append(preds)
    
    y_pred_all = np.concatenate(y_pred_fold)
    y_test_all = y_test_tensor.cpu().numpy()
    
    fold_results.append({
        "fold": fold_num,
        "True_label": y_test_all,
        "Pred_label": y_pred_all,
        "workpiece_usage": usage_test_fold})
    
    print(f"Fold {fold_num} finished\n")

print("Results mit pred_filter")
fno_results = evaluate_folds(fold_results)
fno_results.to_csv('fno_torque-1D_results.csv', index=False)
fno_results

Fold 1von 5
Epoch 1: Train Acc=0.2639, F1=0.2641
Test  Acc=0.2503, F1=0.2154
Neue beste F1: 0.2154
Epoch 2: Train Acc=0.2993, F1=0.2945
Test  Acc=0.3108, F1=0.2923
Neue beste F1: 0.2923
Epoch 3: Train Acc=0.3250, F1=0.3211
Test  Acc=0.3363, F1=0.3072
Neue beste F1: 0.3072
Epoch 4: Train Acc=0.3539, F1=0.3505
Test  Acc=0.3612, F1=0.3321
Neue beste F1: 0.3321
Epoch 5: Train Acc=0.3746, F1=0.3697
Test  Acc=0.3468, F1=0.3208
Epoch 6: Train Acc=0.4026, F1=0.3985
Test  Acc=0.3735, F1=0.3506
Neue beste F1: 0.3506
Epoch 7: Train Acc=0.4138, F1=0.4096
Test  Acc=0.3925, F1=0.3801
Neue beste F1: 0.3801
Epoch 8: Train Acc=0.4343, F1=0.4312
Test  Acc=0.4107, F1=0.3945
Neue beste F1: 0.3945
Epoch 9: Train Acc=0.4566, F1=0.4539
Test  Acc=0.3755, F1=0.3705
Epoch 10: Train Acc=0.4828, F1=0.4809
Test  Acc=0.4135, F1=0.4006
Neue beste F1: 0.4006
Epoch 11: Train Acc=0.5110, F1=0.5096
Test  Acc=0.3968, F1=0.3806
Epoch 12: Train Acc=0.5431, F1=0.5422
Test  Acc=0.3980, F1=0.3947
Epoch 13: Train Acc=0.5732, F

,Labelstrategie,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,F1_macro_mean,F1_macro_std
0,Binary_s1st,82.043868,0.023635,89.227871,0.034242,88.195785,0.022738,88.648260,0.016705
1,Binary_all,78.288000,0.006652,87.448221,0.004416,85.080000,0.013132,86.239649,0.005414
2,Single_s1st,45.306829,0.045998,45.086812,0.029398,42.075756,0.045765,40.178225,0.038506
3,Single_all,39.664000,0.004298,37.823245,0.003719,37.265000,0.005096,37.346859,0.003675
4,Grouped_s1st,55.893390,0.035638,56.721026,0.028814,55.991851,0.043483,55.026169,0.036279
5,Grouped_all,49.744000,0.005838,49.790709,0.004131,49.744000,0.005838,49.644311,0.005121


FNO with features of tsfresh

In [27]:
df_tsfresh = data_v2_df.copy()
if "id" not in df_tsfresh.columns:
    df_tsfresh = data_v2_df.reset_index().rename(columns={"index": "id"})
df_tsfresh.head()

,id,time_values,torque_values,angle_values,gradient_values,step_values,class_values,workpiece_location,workpiece_usage,workpiece_result,scenario_condition,scenario_exception
0,0,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.067, 0.077, 0.126, 0.087, 0.089, 0.097, 0.0...","[0.5, 1.25, 2.25, 3.75, 5.0, 6.25, 7.5, 8.75, ...","[0.0, 0.0, 0.0298, 0.0214, 0.0064, 0.0023, -0....","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,0,OK,normal,0
1,1,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.005, 0.008, 0.061, 0.069, 0.104, 0.124, 0....","[0.0, 0.25, 0.75, 1.5, 2.5, 4.0, 5.25, 6.5, 7....","[0.0, 0.0, 0.0, 0.0, 0.0287, 0.0282, 0.0091, 0...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,0,OK,normal,0
2,2,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, 0.01, 0.039, 0.099, 0.102, 0.087, 0.0...","[0.0, 0.5, 1.25, 2.25, 3.5, 4.75, 6.0, 7.5, 8....","[0.0, 0.0, 0.0, 0.0196, 0.0223, 0.0211, 0.0096...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,1,OK,normal,0
3,3,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.081, 0.059, 0.12, 0.077, 0.043, 0.059, 0.06...","[0.75, 1.75, 2.75, 4.0, 5.25, 6.5, 7.75, 9.25,...","[0.0, 0.0, 0.0261, 0.0207, 0.0, -0.0036, -0.00...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,1,OK,normal,0
4,4,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, -0.0012, 0.0006, 0.0024, 0.0042, 0.00...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 1.5, 2.5, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.025...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,2,OK,normal,0


In [28]:
torque_values = df_tsfresh["torque_values"]
time_values = df_tsfresh["time_values"]
screw_id = df_tsfresh["id"]
y_labels = df_tsfresh["class_values"]

df_long = df_tsfresh.explode('torque_values').reset_index(drop=True)
df_long['time_values'] = df_long.groupby('id').cumcount()
df_long = df_long[['id', 'time_values', 'torque_values']]

df_long['torque_values'] = pd.to_numeric(df_long['torque_values'])
df_long['time_values'] = pd.to_numeric(df_long['time_values']) 
print(df_long.shape) #12500 x 800 = 10mil
df_long.head()

(10000000, 3)


,id,time_values,torque_values
0,0,0,0.067
1,0,1,0.077
2,0,2,0.126
3,0,3,0.087
4,0,4,0.089


In [22]:
print(df_long['time_values'].max())
print(df_long['time_values'].count())

799
10000000


In [23]:
from tsfresh import extract_features, select_features
from tsfresh.utilities.dataframe_functions import impute

X_extr_feat = extract_features(df_long, column_id='id', column_sort='time_values', column_value='torque_values')
impute(X_extr_feat)
y = df_tsfresh.set_index("id")["class_values"]
X_selected = select_features(X_extr_feat,y)

Feature Extraction: 100%|██████████| 30/30 [14:47<00:00, 29.57s/it]  
c:\Users\Patrick\miniconda3\envs\thesis_gpu\Lib\site-packages\tsfresh\utilities\dataframe_functions.py:198: RuntimeWarning: The columns ['torque_values__query_similarity_count__query_None__threshold_0.0'] did not have any finite values. Filling with zeros.
  warnings.warn(


In [35]:
#speichern mit pickle
import pickle
with open('X_selected_tsfresh.pickle','wb') as f:
    pickle.dump(X_selected, f, pickle.HIGHEST_PROTOCOL)
y_tsfresh = y.copy()
with open('y_tsfresh.pickle','wb') as f:
    pickle.dump(y_tsfresh, f, pickle.HIGHEST_PROTOCOL)

In [40]:
#laden mit pickle
"""with open('X_selected_tsfresh.pickle', 'rb') as f:
    # The protocol version used is detected automatically, so we do not
    # have to specify it.
    data = pickle.load(f)"""
    
import pickle   
with open('X_selected_tsfresh.pickle', 'rb') as f:
    # The protocol version used is detected automatically, so we do not
    # have to specify it.
    x_tsfresh = pickle.load(f)
with open('y_tsfresh.pickle', 'rb') as f:
    # The protocol version used is detected automatically, so we do not
    # have to specify it.
    y_tsfresh = pickle.load(f)
print(x_tsfresh.shape)
print(y_tsfresh.shape)
print(y_tsfresh[0])


(12500, 705)
(12500,)
001_control-group


In [42]:
print(x_tsfresh.shape) #macht dann doch gar keinen sinn mehr da keine sequenz mehr sondern einzelne features je sample/screw

(12500, 705)


In [ ]:
print(X_selected.columns[X_selected.isna().all()].tolist())
zero_cols = X_selected.columns[(X_selected == 0).all()].tolist()
print(zero_cols)

[]
[]


In [45]:
from sklearn.preprocessing import LabelEncoder
X = x_tsfresh.to_numpy(dtype=np.float32)
le = LabelEncoder()
y = le.fit_transform(y_tsfresh)


n_classes_tsfresh = len(np.unique(y))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results_tsfresh = []

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    print(f"Fold {fold_num} von 5")

    X_train_fold = X[train_idx]
    X_test_fold = X[test_idx]
    y_train_fold = y[train_idx]
    y_test_fold = y[test_idx]

    scaler = StandardScaler()
    X_train_fold = scaler.fit_transform(X_train_fold)
    X_test_fold = scaler.transform(X_test_fold)

    x_train_tensor = torch.from_numpy(X_train_fold).float().unsqueeze(-1)
    y_train_tensor = torch.from_numpy(y_train_fold).long()

    x_test_tensor = torch.from_numpy(X_test_fold).float().unsqueeze(-1)
    y_test_tensor = torch.from_numpy(y_test_fold).long()

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(x_train_tensor, y_train_tensor),
        batch_size=batch_size,
        shuffle=True
    )
    test_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(x_test_tensor, y_test_tensor),
        batch_size=batch_size,
        shuffle=False
    )

    model = FNO1d(modes, width, n_classes=n_classes_tsfresh).cuda()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

    patience = 30
    best_test_f1 = -np.inf
    patience_counter = 0
    best_model_state = None

    for ep in range(epochs):
        model.train()
        train_cross_entropy = 0.0

        for x, y_batch in train_loader:
            x, y_batch = x.cuda(), y_batch.cuda()
            optimizer.zero_grad()
            out = model(x)
            loss = F.cross_entropy(out, y_batch)
            loss.backward()
            optimizer.step()

            train_cross_entropy += loss.item()
            acc_metric.update(out, y_batch)
            f1_metric.update(out, y_batch)

        epoch_acc = acc_metric.compute()
        epoch_f1 = f1_metric.compute()
        acc_metric.reset()
        f1_metric.reset()

        scheduler.step()
        model.eval()
        test_cross_entropy = 0.0

        with torch.no_grad():
            for x, y_batch in test_loader:
                x, y_batch = x.cuda(), y_batch.cuda()
                out = model(x)
                test_cross_entropy += F.cross_entropy(out, y_batch).item()
                acc_metric.update(out, y_batch)
                f1_metric.update(out, y_batch)

        test_cross_entropy /= len(test_loader)
        epoch_acc = acc_metric.compute()
        epoch_f1 = f1_metric.compute()
        acc_metric.reset()
        f1_metric.reset()

        if float(epoch_f1) > best_test_f1:
            best_test_f1 = float(epoch_f1)
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if best_model_state is not None:
                    model.load_state_dict(best_model_state)
                break

    y_pred_fold = []
    model.eval()
    with torch.no_grad():
        for x, _ in test_loader:
            x = x.cuda()
            out = model(x)
            y_pred_fold.append(out.argmax(dim=1).cpu().numpy())

    y_pred_all = np.concatenate(y_pred_fold)

    fold_results_tsfresh.append({
        "fold": fold_num,
        "True_label": y_test_fold,
        "Pred_label": y_pred_all,
        "workpiece_usage": np.array(df_tsfresh.loc[x_tsfresh.index[test_idx], "workpiece_usage"])
    })
print("Results von FNO mit tsfresh mit pred_filter")
fno_tsfresh_results = evaluate_folds(fold_results_tsfresh)
fno_tsfresh_results.to_csv('fno_tsfresh_results.csv', index=False)
fno_tsfresh_results
    


Fold 1 von 5
Fold 2 von 5
Fold 3 von 5
Fold 4 von 5
Fold 5 von 5
Results von FNO mit tsfresh mit pred_filter


,Labelstrategie,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,F1_macro_mean,F1_macro_std
0,Binary_s1st,79.513207,0.033976,88.370645,0.021587,85.753684,0.035413,86.983915,0.019129
1,Binary_all,74.752000,0.020334,84.451001,0.008962,83.930000,0.036664,84.136143,0.016428
2,Single_s1st,35.615394,0.044846,31.974856,0.046429,33.807699,0.028695,31.310961,0.037913
3,Single_all,30.504000,0.008006,28.533406,0.006558,28.425000,0.008749,28.017231,0.009153
4,Grouped_s1st,45.936145,0.057117,45.753413,0.055087,46.175499,0.059382,44.663497,0.061415
5,Grouped_all,39.544000,0.012953,40.140872,0.008865,39.544000,0.012953,39.554264,0.012458


### Ich bin mir nicht sicher wie sinnvoll das sein soll. Da keine Zeitsequenz zu verarbeiten, aber als solches gebe ich es durch.

## FNO mit Torque + Angle_value

In [29]:
torque = np.array(data_v2_df["torque_values"].tolist(), dtype=np.float32)
angle = np.array(data_v2_df['angle_values'].tolist(), dtype=np.float32)
X_torque_angle = np.stack([torque, angle], axis=2)
X_torque_angle.shape
print(y_labels)

0          001_control-group
1          001_control-group
2          001_control-group
3          001_control-group
4          001_control-group
                ...         
12495    402_surface-chipped
12496    402_surface-chipped
12497    402_surface-chipped
12498    402_surface-chipped
12499    402_surface-chipped
Name: class_values, Length: 12500, dtype: object


In [37]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
le = LabelEncoder()
y = le.fit_transform(y)

n_classes_tsfresh = len(np.unique(y))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results_torque_angle = []

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X_torque_angle, y), start=1):
    print(f"Fold {fold_num} von 5")

    X_train_fold = X_torque_angle[train_idx]
    X_test_fold = X_torque_angle[test_idx]
    y_train_fold = y[train_idx]
    y_test_fold = y[test_idx]

    scaler = StandardScaler()
    orig_shape = X_train_fold.shape
    orig_shape_test = X_test_fold.shape
    X_train_fold = scaler.fit_transform(X_train_fold.reshape(X_train_fold.shape[0], -1)) #ValueError: Found array with dim 3, while dim <= 2 is required by StandardScaler. umskallieren zu 2d und dann wieder zurück auf 3d
    X_train_fold = X_train_fold.reshape(orig_shape) 
    X_test_fold = scaler.transform(X_test_fold.reshape(X_test_fold.shape[0], -1)) #durch -1 berechnet numpy automatisch die anzahl der spalten also 2*num features. shape[2] geht nicht da sonst nur ein feature
    X_test_fold = X_test_fold.reshape(orig_shape_test)
    
    x_train_tensor = torch.from_numpy(X_train_fold).float()
    y_train_tensor = torch.from_numpy(y_train_fold).long()

    x_test_tensor = torch.from_numpy(X_test_fold).float()
    y_test_tensor = torch.from_numpy(y_test_fold).long()

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(x_train_tensor, y_train_tensor),
        batch_size=batch_size,
        shuffle=True
    )
    test_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(x_test_tensor, y_test_tensor),
        batch_size=batch_size,
        shuffle=False
    )

    model = FNO2d(modes, width, n_classes=n_classes_tsfresh, input_channels=2).cuda()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

    patience = 30
    best_test_f1 = -np.inf
    patience_counter = 0
    best_model_state = None

    for ep in range(epochs):
        model.train()
        train_cross_entropy = 0.0

        for x, y_batch in train_loader:
            x, y_batch = x.cuda(), y_batch.cuda()
            optimizer.zero_grad()
            out = model(x)
            loss = F.cross_entropy(out, y_batch)
            loss.backward()
            optimizer.step()

            train_cross_entropy += loss.item()
            acc_metric.update(out, y_batch)
            f1_metric.update(out, y_batch)

        epoch_acc = acc_metric.compute()
        epoch_f1 = f1_metric.compute()
        acc_metric.reset()
        f1_metric.reset()

        scheduler.step()
        model.eval()
        test_cross_entropy = 0.0

        with torch.no_grad():
            for x, y_batch in test_loader:
                x, y_batch = x.cuda(), y_batch.cuda()
                out = model(x)
                test_cross_entropy += F.cross_entropy(out, y_batch).item()
                acc_metric.update(out, y_batch)
                f1_metric.update(out, y_batch)

        test_cross_entropy /= len(test_loader)
        epoch_acc = acc_metric.compute()
        epoch_f1 = f1_metric.compute()
        acc_metric.reset()
        f1_metric.reset()

        if float(epoch_f1) > best_test_f1:
            best_test_f1 = float(epoch_f1)
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if best_model_state is not None:
                    model.load_state_dict(best_model_state)
                break

    y_pred_fold = []
    model.eval()
    with torch.no_grad():
        for x, _ in test_loader:
            x = x.cuda()
            out = model(x)
            y_pred_fold.append(out.argmax(dim=1).cpu().numpy())

    y_pred_all = np.concatenate(y_pred_fold)

    fold_results_torque_angle.append({
        "fold": fold_num,
        "True_label": y_test_fold,
        "Pred_label": y_pred_all,
        "workpiece_usage": usage[test_idx]
    })
    
print("Results von FNO 2D mit torque_angle mit pred_filter")
fno_torque_angle_results = evaluate_folds(fold_results_torque_angle)
fno_torque_angle_results.to_csv('fno_torque_angle_results.csv', index=False)
fno_torque_angle_results
    


Fold 1 von 5
Fold 2 von 5
Fold 3 von 5
Fold 4 von 5
Fold 5 von 5
Results von FNO 2D mit torque_angle mit pred_filter


,Labelstrategie,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,F1_macro_mean,F1_macro_std
0,Binary_s1st,83.541770,0.011562,87.533401,0.028303,92.685120,0.022861,89.968347,0.007946
1,Binary_all,78.576000,0.010985,85.894331,0.014299,87.690000,0.026509,86.739586,0.008641
2,Single_s1st,46.186340,0.062901,46.297197,0.045234,43.684991,0.058164,42.657109,0.055822
3,Single_all,37.904000,0.008438,36.416996,0.008777,36.030000,0.004554,35.835597,0.007025
4,Grouped_s1st,58.023986,0.052685,59.185011,0.062017,57.681392,0.060089,56.932958,0.062047
5,Grouped_all,48.336000,0.012796,48.434140,0.011029,48.336000,0.012796,48.103998,0.012718


## FNO mit torque und Work rotational force (angle*torque)

In [38]:
W = torque * angle
X_torque_W = np.stack([torque, W], axis=2)
print(X_torque_W.shape)

(12500, 800, 2)


In [39]:
le = LabelEncoder()
y = le.fit_transform(y)

n_classes_tsfresh = len(np.unique(y))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results_torque_W = []

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X_torque_W, y), start=1):
    print(f"Fold {fold_num} von 5")

    X_train_fold = X_torque_W[train_idx]
    X_test_fold = X_torque_W[test_idx]
    y_train_fold = y[train_idx]
    y_test_fold = y[test_idx]

    scaler = StandardScaler()
    orig_shape = X_train_fold.shape
    orig_shape_test = X_test_fold.shape
    X_train_fold = scaler.fit_transform(X_train_fold.reshape(X_train_fold.shape[0], -1)) #ValueError: Found array with dim 3, while dim <= 2 is required by StandardScaler. umskallieren zu 2d und dann wieder zurück auf 3d
    X_train_fold = X_train_fold.reshape(orig_shape) 
    X_test_fold = scaler.transform(X_test_fold.reshape(X_test_fold.shape[0], -1)) #durch -1 berechnet numpy automatisch die anzahl der spalten also 2*num features. shape[2] geht nicht da sonst nur ein feature
    X_test_fold = X_test_fold.reshape(orig_shape_test)
    
    x_train_tensor = torch.from_numpy(X_train_fold).float()
    y_train_tensor = torch.from_numpy(y_train_fold).long()

    x_test_tensor = torch.from_numpy(X_test_fold).float()
    y_test_tensor = torch.from_numpy(y_test_fold).long()

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(x_train_tensor, y_train_tensor),
        batch_size=batch_size,
        shuffle=True
    )
    test_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(x_test_tensor, y_test_tensor),
        batch_size=batch_size,
        shuffle=False
    )

    model = FNO2d(modes, width, n_classes=n_classes_tsfresh, input_channels=2).cuda()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

    patience = 30
    best_test_f1 = -np.inf
    patience_counter = 0
    best_model_state = None

    for ep in range(epochs):
        model.train()
        train_cross_entropy = 0.0

        for x, y_batch in train_loader:
            x, y_batch = x.cuda(), y_batch.cuda()
            optimizer.zero_grad()
            out = model(x)
            loss = F.cross_entropy(out, y_batch)
            loss.backward()
            optimizer.step()

            train_cross_entropy += loss.item()
            acc_metric.update(out, y_batch)
            f1_metric.update(out, y_batch)

        epoch_acc = acc_metric.compute()
        epoch_f1 = f1_metric.compute()
        acc_metric.reset()
        f1_metric.reset()

        scheduler.step()
        model.eval()
        test_cross_entropy = 0.0

        with torch.no_grad():
            for x, y_batch in test_loader:
                x, y_batch = x.cuda(), y_batch.cuda()
                out = model(x)
                test_cross_entropy += F.cross_entropy(out, y_batch).item()
                acc_metric.update(out, y_batch)
                f1_metric.update(out, y_batch)

        test_cross_entropy /= len(test_loader)
        epoch_acc = acc_metric.compute()
        epoch_f1 = f1_metric.compute()
        acc_metric.reset()
        f1_metric.reset()

        if float(epoch_f1) > best_test_f1:
            best_test_f1 = float(epoch_f1)
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if best_model_state is not None:
                    model.load_state_dict(best_model_state)
                break

    y_pred_fold = []
    model.eval()
    with torch.no_grad():
        for x, _ in test_loader:
            x = x.cuda()
            out = model(x)
            y_pred_fold.append(out.argmax(dim=1).cpu().numpy())

    y_pred_all = np.concatenate(y_pred_fold)

    fold_results_torque_W.append({
        "fold": fold_num,
        "True_label": y_test_fold,
        "Pred_label": y_pred_all,
        "workpiece_usage": usage[test_idx]
    })
    
print("Results von FNO 2D mit torque_W mit pred_filter")
fno_torque_W_results = evaluate_folds(fold_results_torque_W)
fno_torque_W_results.to_csv('fno_torque_W_results.csv', index=False)
fno_torque_W_results
    


Fold 1 von 5
Fold 2 von 5
Fold 3 von 5
Fold 4 von 5
Fold 5 von 5
Results von FNO 2D mit torque_W mit pred_filter


,Labelstrategie,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,F1_macro_mean,F1_macro_std
0,Binary_s1st,79.573311,0.020739,87.270939,0.029241,87.186843,0.018990,87.178509,0.013209
1,Binary_all,78.424000,0.009434,87.303900,0.011732,85.500000,0.017880,86.370922,0.006882
2,Single_s1st,45.212974,0.017212,45.214151,0.040200,42.489328,0.031247,42.336821,0.027328
3,Single_all,39.520000,0.004996,37.583038,0.008616,37.505000,0.007269,37.348727,0.007505
4,Grouped_s1st,59.252426,0.022710,59.478642,0.019685,58.957205,0.019834,58.370654,0.020064
5,Grouped_all,49.784000,0.005418,50.017470,0.004774,49.784000,0.005418,49.703757,0.003638


In [22]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
def pred_filter(fold_results, label="binary", usage1st=True):
    metriken = {"Accuracy": [], "Precision": [], "Recall": [], "F1_macro": []}
    if label == "binary":
        average_method = "binary"
    elif label in ["single", "grouped"]:
        average_method = "macro"
    
    for fold in fold_results:
        y_val = fold["True_label"]
        y_pred = fold["Pred_label"]
        usage_val = fold["workpiece_usage"]
        if usage1st == True:
            usage_filter = usage_val == 0
            y_val_filtered = y_val[usage_filter]
            y_pred_filtered = y_pred[usage_filter]
        else:
            y_val_filtered = y_val
            y_pred_filtered = y_pred
        if label == "binary":
            y_val_binary = []
            y_pred_binary = []
            for l in y_val_filtered:
                if str(l).startswith('0'):
                    y_val_binary.append(0)
                else:
                    y_val_binary.append(1)
            for l in y_pred_filtered:
                if str(l).startswith('0'):
                    y_pred_binary.append(0)
                else:
                    y_pred_binary.append(1)
            y_val_filtered = np.array(y_val_binary)
            y_pred_filtered = np.array(y_pred_binary)
        elif label == "grouped":
            y_val_group = []
            y_pred_group = []
            for l in y_val_filtered:
                if str(l).startswith('0'):
                    y_val_group.append(0)
                elif str(l).startswith('1'):
                    y_val_group.append(1)
                elif str(l).startswith('2') or str(l).startswith('3'):
                    y_val_group.append(2)
                elif str(l).startswith('4') or str(l).startswith('5'):
                    y_val_group.append(3)
                elif str(l).startswith('6') or str(l).startswith('7'):
                    y_val_group.append(4)
            for l in y_pred_filtered:
                if str(l).startswith('0'):
                    y_pred_group.append(0)
                elif str(l).startswith('1'):
                    y_pred_group.append(1)
                elif str(l).startswith('2') or str(l).startswith('3'):
                    y_pred_group.append(2)
                elif str(l).startswith('4') or str(l).startswith('5'):
                    y_pred_group.append(3)
                elif str(l).startswith('6') or str(l).startswith('7'):
                    y_pred_group.append(4)
            y_val_filtered = np.array(y_val_group)
            y_pred_filtered = np.array(y_pred_group)
        elif label == "single":
            y_val_filtered = np.array(y_val_filtered)
            y_pred_filtered = np.array(y_pred_filtered)
        metriken["Accuracy"].append(accuracy_score(y_val_filtered, y_pred_filtered))
        metriken["Precision"].append(precision_score(y_val_filtered, y_pred_filtered, average=average_method, zero_division=0))
        metriken["Recall"].append(recall_score(y_val_filtered, y_pred_filtered, average=average_method, zero_division=0))
        metriken["F1_macro"].append(f1_score(y_val_filtered, y_pred_filtered, average=average_method, zero_division=0))
    
    stats = {"Accuracy_mean": np.mean(metriken["Accuracy"]), "Accuracy_std": np.std(metriken["Accuracy"]), "Precision_mean": np.mean(metriken["Precision"]), "Precision_std": np.std(metriken["Precision"]), "Recall_mean": np.mean(metriken["Recall"]), "Recall_std": np.std(metriken["Recall"]), "F1_macro_mean": np.mean(metriken["F1_macro"]), "F1_macro_std": np.std(metriken["F1_macro"])}
    for i in stats:
        if i != "Accuracy_std" and i != "Precision_std" and i != "Recall_std" and i != "F1_macro_std":
            stats[i] *= 100
    df = pd.DataFrame([stats])
    return df


def evaluate_folds(fold_results):
    df_binary_s1st = pred_filter(fold_results, label="binary", usage1st=True)
    df_binary_all = pred_filter(fold_results, label="binary", usage1st=False)
    df_single_s1st = pred_filter(fold_results, label="single", usage1st=True)
    df_single_all = pred_filter(fold_results, label="single", usage1st=False)
    df_grouped_s1st = pred_filter(fold_results, label="grouped", usage1st=True)
    df_grouped_all = pred_filter(fold_results, label="grouped", usage1st=False)
    
    results = pd.concat([df_binary_s1st, df_binary_all, df_single_s1st, df_single_all, df_grouped_s1st, df_grouped_all], ignore_index=True)

    results.insert(0, 'Labelstrategie', ['Binary_s1st', 'Binary_all', 'Single_s1st', 'Single_all','Grouped_s1st', 'Grouped_all'])
    return results

In [28]:
pred = []
model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.cuda(), y.cuda()
        out = model(x)
        preds = out.argmax(dim=1).cpu().numpy()
        pred.append(preds)

y_pred_all = np.concatenate(pred)
y_test_all = y_test_tensor.cpu().numpy()
usage_test_all = np.array(data_v2_df['workpiece_usage'].tolist())[split_idx:]


fold_results = [{
    "True_label": y_test_all,
    "Pred_label": y_pred_all,
    "workpiece_usage": usage_test_all
}]

evaluate_folds(fold_results)

,Labelstrategie,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,F1_macro_mean,F1_macro_std
0,Binary_s1st,78.034682,0.0,87.313433,0.0,84.782609,0.0,86.029412,0.0
1,Binary_all,78.651429,0.0,87.027962,0.0,86.183271,0.0,86.603557,0.0
2,Single_s1st,37.572254,0.0,34.710292,0.0,34.339123,0.0,34.052257,0.0
3,Single_all,40.365714,0.0,38.219713,0.0,38.376939,0.0,38.183187,0.0
4,Grouped_s1st,50.289017,0.0,50.490866,0.0,50.829386,0.0,50.078742,0.0
5,Grouped_all,50.560000,0.0,50.899666,0.0,50.613415,0.0,50.675046,0.0
